# ДЗ 2: 20 000 train-пар, прежний Transformer

**Гипотеза:** более разнообразный train поможет модели выучить общие шаблоны кода и связь с вопросом. Проверка запоминания уже успешна: прежняя модель точно воспроизвела 16/16 программ с EOS на шаге 2250. Теперь обучаем новую модель **с нуля**, без загрузки тех весов.

В Colab выбери **T4** и выполни все ячейки по порядку в новой сессии. Если отсутствует PyArrow, выполни отдельную ячейку `%pip install pyarrow`. Скачивается только первый разрешённый train-файл. Программы и таргеты не редактируются; test не скачивается и не генерируется в этом опыте.

Берём первые **20 000** подходящих train-пар из того же shard: вопрос ≤512 UTF-8-байтов, код <512 байтов, прежний split по целому вопросу. Первые 512 совпадают с предыдущим train. Прежние **64 validation-пары** остаются теми же; все hash-heldout вопросы исключены из train. Выборка содержит 19 960 разных train-вопросов. Это расширение одного shard, а не случайная выборка всего датасета.

Прежние UTF-8-байты, Transformer 2+2/128 с 4 головами, FFN 512, ReLU/Post-LN, обучаемые позиции, dropout 0.1, FP32, AdamW LR 0.001 / weight decay 0.01, clipping 1.0, seed 42 и batch 8. Единственное изменение для сравнения на 2000 шагах — train 512 → 20 000. Затем продолжение до 5000 и 10 000 отдельно проверяет длительность: 0.8 / 2 / 4 предъявления на пару в среднем, не точные эпохи.

Каждые 50 шагов — loss на прежних 16 validation-парах. На **2000 / 5000 / 10 000** — token-weighted loss и greedy на прежних первых 64 train-парах и всех 64 validation-парах; AST, `solve`, EOS, длины и совпадение с единственным сохранённым кодом. Печатаются первые 3 генерации каждой части. Exact/AST не являются функциональной оценкой; программы не исполняются.

Время цикла обучения измеряется **без validation, greedy и сохранения**: внутри таймера подготовка batch/переносы, forward/backward, clipping и AdamW. На контрольных точках отдельно измеряется greedy 64 вопросов с batch 32 и cap 512 после короткого warm-up. Его таймер включает кодирование, переносы, генерацию, декодирование и AST; loss и checkpoint исключены. Это диагностика скорости, не проверка 1919 test-ответов за 180 с. Накладные расходы и общий активный wall time записываются отдельно. Число токенов включает EOS, PAD исключён; EOS на последней разрешённой позиции отдельно виден через EOS + reached-cap.

Скачай **`hw02-byte-20k-results.json`** и сохрани notebook с outputs. Checkpoints: **`hw02-byte-20k-step-2000.pt`**, **`hw02-byte-20k-step-5000.pt`**, **`hw02-byte-20k-step-10000.pt`**. Дополнительно `hw02-byte-20k-latest.pt` обновляется каждые 1000 шагов; в нём есть optimizer и RNG для будущего восстановления. Результаты обновляются после логирования и каждой диагностики. Не запускай тренировочную ячейку повторно на уже обученном optimizer; для нового опыта используй новую сессию и все ячейки по порядку.

На момент подготовки 10 000 шагов и T4-прогон не выполнены. Локальная CPU-проверка кода не является результатом качества. Исторические числа 512-парного опыта ниже получены пользователем на T4; новая среда фиксируется при запуске.

Источники: [ДЗ 2](https://github.com/ashaba1in/hse-nlp/blob/7f837940c4af9a686fca7bff5f02345218b6fe80/homeworks/hw2_code_generation.md), [датасет](https://huggingface.co/datasets/SlavaYus/tinysms_qa_12m/blob/b2724689ff079e9c22efd548fe134bfb29036dbb/README.md), [лекция 2](https://github.com/ashaba1in/hse-nlp/blob/24a73bd327d699039e49e8e169bbce59ac5bbb03/week02_transformer/lecture2.pdf), [лекция 3](https://github.com/ashaba1in/hse-nlp/blob/24a73bd327d699039e49e8e169bbce59ac5bbb03/week03_bert_gpt/lecture3.pdf). Числа 20 000 и контрольные бюджеты — наши решения.


In [ ]:
# HW2_BYTE_20K: same model and optimization; more allowed training pairs.
import ast
import hashlib
import json
import math
import random
import time
from collections import Counter
from pathlib import Path
from urllib.request import urlretrieve

import torch
from torch import nn
from torch.nn.utils.rnn import pad_sequence
import pyarrow.parquet as pq

SEED, TRAIN_SIZE, BATCH_SIZE = 42, 20000, 8
EVALUATION_STEPS = (2000, 5000, 10000)
TOTAL_STEPS, LOG_EVERY, CHECKPOINT_EVERY = EVALUATION_STEPS[-1], 50, 1000
D_MODEL, N_LAYERS, MAX_POSITIONS = 128, 2, 2048
PAD, BOS, EOS, VOCAB = 0, 1, 2, 259
GENERATION_CAP, INFERENCE_BATCH = 512, 32
random.seed(SEED)
torch.manual_seed(SEED)
assert torch.cuda.is_available(), 'Enable GPU in Colab before running this notebook'
device = torch.device('cuda')
print('Device:', torch.cuda.get_device_name(0))
print('PyTorch:', torch.__version__)

RESULTS_FILE = Path('hw02-byte-20k-results.json')
LATEST_CHECKPOINT = 'hw02-byte-20k-latest.pt'
def sync():
    if device.type == 'cuda':
        torch.cuda.synchronize()
def data_hash(rows):
    return hashlib.sha256(json.dumps(rows, ensure_ascii=False, sort_keys=True).encode()).hexdigest()
def save_results():
    RESULTS_FILE.write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding='utf-8')

previous_512_run = {
    'origin': 'saved user T4 run, PyTorch 2.11.0+cu130; not measured by this notebook',
    'training_examples': 512, 'steps': 2000,
    'validation_loss_16': 1.7727019786834717,
    'train_64_loss': 1.3539199228447052, 'validation_64_loss': 1.8114380311731377,
    'train_64_ast_count': 3, 'validation_64_ast_count': 2,
    'train_64_eos_count': 22, 'validation_64_eos_count': 31,
    'validation_64_greedy_seconds': 12.581568289000018,
    'training_loop_seconds_rounded_including_periodic_validation': 51.45,
}


In [ ]:
# 1. Same first shard/filter/split; expand only the training list.
revision = 'b2724689ff079e9c22efd548fe134bfb29036dbb'
base_url = f'https://huggingface.co/datasets/SlavaYus/tinysms_qa_12m/resolve/{revision}'
cache = Path('hw02-data')
cache.mkdir(exist_ok=True)
train_file = cache / 'train.parquet'
data_started = time.perf_counter()
if not train_file.exists():
    urlretrieve(f'{base_url}/data/train/part-00000.parquet', train_file)
def is_validation(question):
    key = hashlib.sha256(f'{SEED}:{question}'.encode()).digest()
    return int.from_bytes(key[:8], 'big') % 10 == 0
def eligible(row):
    return len(row['question'].encode()) <= 512 and len(row['code'].encode()) < 512

train, train_row_indices, validation = [], [], None
offset, rejected_length, excluded_heldout = 0, 0, 0
for batch in pq.ParquetFile(train_file).iter_batches(batch_size=4096):
    rows = batch.to_pylist()
    if validation is None:
        validation = [r for r in rows if eligible(r) and is_validation(r['question'])][:64]
    for index, row in enumerate(rows):
        if not eligible(row):
            rejected_length += 1
            continue
        if is_validation(row['question']):
            excluded_heldout += 1
            continue
        train.append(row)
        train_row_indices.append(offset + index)
        if len(train) == TRAIN_SIZE:
            break
    if len(train) == TRAIN_SIZE:
        break
    offset += len(rows)
assert len(train) == TRAIN_SIZE and len(validation) == 64
train_hash, validation_hash = data_hash(train), data_hash(validation)
prefix512_hash = data_hash(train[:512])
assert train_hash == '1304183967f02f01097d76c12fb24df30916a4ec44e54833cd3509264359300e'
assert prefix512_hash == '3801627c7dcaae0c82c6694e0aa8514ca8ced044c07c7f30e779bb6a3130bc35'
assert validation_hash == '0edf2b479ebdc103ec05efc704c7bb0ac8084e41a8a44b58afef224702563050'
assert all(not is_validation(r['question']) for r in train)
question_counts = Counter(r['question'] for r in train)
assert not (set(question_counts) & {r['question'] for r in validation})
train_diagnostic = train[:64]  # Same 64 train pairs used in the 512-pair experiment.
data_info = {
    'revision': revision, 'shard': 0, 'training_examples': len(train),
    'unique_train_questions': len(question_counts),
    'duplicate_rows_beyond_first_question': len(train) - len(question_counts),
    'max_question_multiplicity': max(question_counts.values()),
    'validation_examples': len(validation),
    'train_sha256': train_hash, 'validation_sha256': validation_hash,
    'train_prefix512_sha256': prefix512_hash,
    'train_row_indices_sha256': data_hash(train_row_indices),
    'first_train_row_index': train_row_indices[0], 'last_train_row_index': train_row_indices[-1],
    'raw_rows_scanned': train_row_indices[-1] + 1,
    'rejected_length_before_last_selected_row': rejected_length,
    'excluded_hash_heldout_before_last_selected_row': excluded_heldout,
    'selection': 'first 20000 eligible non-heldout rows of shard 0; question<=512 bytes, code<512 bytes',
    'question_overlap_train_validation': 0,
    'download_and_selection_seconds': time.perf_counter() - data_started,
}
print('DATA', json.dumps(data_info), flush=True)


In [ ]:
# 2. Byte encoding: reversible, no tokenizer training or external vocabulary.
def encode(text):
    return [b + 3 for b in text.encode('utf-8')]
def decode(ids):
    return bytes(i - 3 for i in ids if 3 <= i < VOCAB).decode('utf-8', errors='replace')
def source_batch(questions):
    return pad_sequence([torch.tensor([BOS] + encode(q) + [EOS]) for q in questions],
                        batch_first=True, padding_value=PAD).to(device)
def training_batch(examples):
    src = source_batch([r['question'] for r in examples])
    codes = [encode(r['code']) for r in examples]
    tgt = pad_sequence([torch.tensor([BOS] + c) for c in codes], batch_first=True, padding_value=PAD).to(device)
    labels = pad_sequence([torch.tensor(c + [EOS]) for c in codes], batch_first=True, padding_value=PAD).to(device)
    return src, tgt, labels



In [ ]:
# 3. Standard encoder-decoder: 2 + 2 layers, learned positions, ReLU, Post-LN.
class SimpleTransformer(nn.Module):
    def __init__(self):
        super().__init__()
        self.tokens = nn.Embedding(VOCAB, D_MODEL, padding_idx=PAD)
        self.positions = nn.Embedding(MAX_POSITIONS, D_MODEL)
        self.transformer = nn.Transformer(d_model=D_MODEL, nhead=4,
            num_encoder_layers=N_LAYERS, num_decoder_layers=N_LAYERS,
            dim_feedforward=4 * D_MODEL, dropout=0.1, batch_first=True)
        self.transformer.encoder.enable_nested_tensor = False
        self.transformer.encoder.use_nested_tensor = False
        self.head = nn.Linear(D_MODEL, VOCAB)
    def embed(self, ids):
        if ids.shape[1] > MAX_POSITIONS:
            raise ValueError('Sequence too long; no silent truncation')
        pos = torch.arange(ids.shape[1], device=ids.device)
        return self.tokens(ids) * math.sqrt(D_MODEL) + self.positions(pos)
    def decode_prefix(self, tgt, memory, source_padding):
        mask = torch.ones(tgt.shape[1], tgt.shape[1], device=tgt.device, dtype=torch.bool).triu(1)
        return self.transformer.decoder(self.embed(tgt), memory, tgt_mask=mask,
            tgt_key_padding_mask=tgt.eq(PAD), memory_key_padding_mask=source_padding)
    def forward(self, src, tgt):
        memory = self.transformer.encoder(self.embed(src), src_key_padding_mask=src.eq(PAD))
        return self.head(self.decode_prefix(tgt, memory, src.eq(PAD)))
    @torch.inference_mode()
    def generate(self, src, max_new_tokens=512):
        self.eval()
        memory = self.transformer.encoder(self.embed(src), src_key_padding_mask=src.eq(PAD))
        tgt = torch.full((len(src), 1), BOS, device=src.device, dtype=torch.long)
        done = torch.zeros(len(src), device=src.device, dtype=torch.bool)
        for _ in range(max_new_tokens):
            hidden = self.decode_prefix(tgt, memory, src.eq(PAD))
            logits = self.head(hidden[:, -1])
            logits[:, [PAD, BOS]] = -torch.inf
            token = logits.argmax(-1).masked_fill(done, PAD)
            done |= token.eq(EOS)
            tgt = torch.cat([tgt, token[:, None]], dim=1)
            if bool(done.all()):
                break
        return tgt[:, 1:]

model = SimpleTransformer().to(device)
criterion = nn.CrossEntropyLoss(ignore_index=PAD)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
print('Parameters:', sum(p.numel() for p in model.parameters()))



In [ ]:
# 4. Fixed held-out loss and fixed train/validation greedy diagnostics.
@torch.inference_mode()
def evaluate_loss(rows):
    model.eval()
    loss_sum, label_count = 0.0, 0
    for offset in range(0, len(rows), 16):
        src, tgt, labels = training_batch(rows[offset:offset + 16])
        logits = model(src, tgt)
        loss_sum += nn.functional.cross_entropy(logits.reshape(-1, VOCAB), labels.reshape(-1),
                                               ignore_index=PAD, reduction='sum').item()
        label_count += labels.ne(PAD).sum().item()
    return loss_sum / label_count

def inspect_ids(row, ids):
    active = [i for i in ids if i != PAD]
    code = decode(active)
    try:
        tree = ast.parse(code)
        parses = True
        defines_solve = any(isinstance(node, ast.FunctionDef) and node.name == 'solve'
                            for node in tree.body)
    except (SyntaxError, ValueError):
        parses, defines_solve = False, False
    return {'question': row['question'], 'reference_code': row['code'], 'generated_code': code,
        'generated_tokens': len(active), 'expected_tokens_including_eos': len(encode(row['code'])) + 1,
        'eos_emitted': EOS in active,
        'early_eos': EOS in active and active.index(EOS) + 1 < GENERATION_CAP,
        'reached_cap': len(active) == GENERATION_CAP,
        'parses_as_python': parses, 'defines_solve': defines_solve,
        'exact_code_match': code == row['code'],
        'exact_tokens_with_eos': active == encode(row['code']) + [EOS]}

@torch.inference_mode()
def inspect_split(name, rows):
    model.eval()
    fixed_loss = evaluate_loss(rows)  # Excluded from the generation timer.
    sync()
    if device.type == 'cuda':
        torch.cuda.reset_peak_memory_stats()
    generation_started = time.perf_counter()
    examples = []
    for offset in range(0, len(rows), INFERENCE_BATCH):
        batch = rows[offset:offset + INFERENCE_BATCH]
        generated = model.generate(source_batch([r['question'] for r in batch]),
                                   max_new_tokens=GENERATION_CAP).cpu().tolist()
        examples.extend(inspect_ids(row, ids) for row, ids in zip(batch, generated))
    sync()
    seconds = time.perf_counter() - generation_started
    lengths = [e['generated_tokens'] for e in examples]
    summary = {'split': name, 'examples': len(rows), 'loss': fixed_loss,
        'mean_generated_tokens': sum(lengths) / len(lengths),
        'min_generated_tokens': min(lengths), 'max_generated_tokens': max(lengths),
        'seconds': seconds, 'answers_per_second': len(rows) / seconds,
        'generated_tokens': sum(lengths), 'tokens_per_second': sum(lengths) / seconds,
        'batch_size_inference': INFERENCE_BATCH, 'max_new_tokens': GENERATION_CAP,
        'timer_scope': 'encoding, transfers, greedy, CPU decoding and AST/exact checks; loss/checkpoint excluded',
        'peak_gpu_memory_mib': torch.cuda.max_memory_allocated() / 2**20 if device.type == 'cuda' else None}
    for metric in ['eos_emitted', 'early_eos', 'reached_cap', 'parses_as_python', 'defines_solve',
                   'exact_code_match', 'exact_tokens_with_eos']:
        summary[metric + '_count'] = sum(e[metric] for e in examples)
        summary[metric + '_fraction'] = summary[metric + '_count'] / len(examples)
    print('DIAGNOSTICS', json.dumps(summary), flush=True)
    for example in examples[:3]:
        print('EXAMPLE', name, json.dumps(example, ensure_ascii=False), flush=True)
    return {'summary': summary, 'examples': examples}

def save_checkpoint(step):
    state = {'model': {k: v.cpu() for k, v in model.state_dict().items()},
        'optimizer': optimizer.state_dict(), 'steps': step, 'config': config,
        'data_revision': revision, 'train_sha256': train_hash, 'validation_sha256': validation_hash,
        'python_random_state': random.getstate(), 'torch_rng_state': torch.get_rng_state(),
        'cuda_rng_state_all': torch.cuda.get_rng_state_all() if device.type == 'cuda' else []}
    torch.save(state, LATEST_CHECKPOINT)
    result['latest_checkpoint_step'] = step
    if step in EVALUATION_STEPS:
        name = f'hw02-byte-20k-step-{step}.pt'
        torch.save(state, name)
        result['checkpoint_files'][str(step)] = name


In [ ]:
# 5. Training is timed in blocks; evaluation/logging/checkpoints are outside it.
assert not optimizer.state, 'Training already ran: use a fresh runtime instead of restarting this cell'
assert next(model.parameters()).dtype == torch.float32
config = {'seed': SEED, 'batch_size_train': BATCH_SIZE, 'total_steps': TOTAL_STEPS,
    'evaluation_steps': list(EVALUATION_STEPS), 'd_model': D_MODEL,
    'encoder_layers': N_LAYERS, 'decoder_layers': N_LAYERS, 'heads': 4,
    'ffn_dim': 4 * D_MODEL, 'dropout': 0.1, 'norm_first': False, 'lr': 0.001,
    'optimizer': 'AdamW', 'weight_decay': optimizer.param_groups[0]['weight_decay'],
    'clip_norm': 1.0, 'scheduler': None, 'tokenizer': 'UTF-8 bytes',
    'max_positions': MAX_POSITIONS, 'generation_cap': GENERATION_CAP,
    'batch_size_inference': INFERENCE_BATCH}
result = {'experiment': 'byte-20k-pairs', 'config': config, 'data': data_info,
    'gpu': torch.cuda.get_device_name(0) if device.type == 'cuda' else 'CPU local check',
    'torch_version': torch.__version__, 'dtype': str(next(model.parameters()).dtype),
    'parameters': sum(p.numel() for p in model.parameters()), 'actual_steps': 0,
    'training_log': [], 'diagnostics': {}, 'checkpoint_files': {},
    'previous_512_pair_run': previous_512_run,
    'training_seconds_excluding_evaluation_logging_and_checkpoints': 0.0,
    'training_timer_scope': 'batch construction, transfers, forward/backward, clipping and AdamW; block-end CUDA sync',
    'run_finished': False, 'functional_quality_evaluated': False,
    'generalization_loss_and_code_diagnostics_evaluated': False, 'test_evaluated': False}
sync()
run_started = time.perf_counter()
initial = {'step': 0, 'train_batch_loss': None, 'validation_loss_16': evaluate_loss(validation[:16]),
           'training_seconds_excluding_evaluation_logging_and_checkpoints': 0.0}
result['training_log'].append(initial)
print('LOSS', json.dumps(initial), flush=True)
save_results()
sync()
block_started = time.perf_counter()
training_seconds = 0.0

for step in range(1, TOTAL_STEPS + 1):
    model.train()
    src, tgt, labels = training_batch(random.sample(train, BATCH_SIZE))
    logits = model(src, tgt)
    loss = criterion(logits.reshape(-1, VOCAB), labels.reshape(-1))
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()
    if step == 1 or step % LOG_EVERY == 0 or step in EVALUATION_STEPS or step == TOTAL_STEPS:
        sync()
        training_seconds += time.perf_counter() - block_started
        result['actual_steps'] = step
        result['training_seconds_excluding_evaluation_logging_and_checkpoints'] = training_seconds
        entry = {'step': step, 'train_batch_loss': loss.item(),
            'validation_loss_16': evaluate_loss(validation[:16]),
            'training_seconds_excluding_evaluation_logging_and_checkpoints': training_seconds,
            'training_updates_per_second_excluding_evaluation_logging_and_checkpoints': step / training_seconds,
            'mean_training_pair_presentations': step * BATCH_SIZE / len(train)}
        result['training_log'].append(entry)
        print('LOSS', json.dumps(entry), flush=True)
        result['active_wall_seconds_including_diagnostics_and_checkpointing'] = time.perf_counter() - run_started
        save_results()
        if step in EVALUATION_STEPS:
            # Greedy warm-up and checks are outside the training timer; no updates/RNG reseeding.
            model.generate(source_batch([validation[0]['question']]), max_new_tokens=8)
            result['diagnostics'][str(step)] = {}
            for name, fixed_rows in [('train_64', train_diagnostic), ('validation_64', validation)]:
                result['diagnostics'][str(step)][name] = inspect_split(name, fixed_rows)
                save_results()
            result['generalization_loss_and_code_diagnostics_evaluated'] = True
        if step % CHECKPOINT_EVERY == 0 or step in EVALUATION_STEPS or step == TOTAL_STEPS:
            save_checkpoint(step)
            print('CHECKPOINT', step, LATEST_CHECKPOINT, flush=True)
        result['active_wall_seconds_including_diagnostics_and_checkpointing'] = time.perf_counter() - run_started
        save_results()
        sync()
        block_started = time.perf_counter()

sync()
result['run_finished'] = True
result['active_wall_seconds_including_diagnostics_and_checkpointing'] = time.perf_counter() - run_started
save_results()


In [ ]:
# 6. One row per checkpoint; compare only common datasets/metrics.
for point in EVALUATION_STEPS:
    checked = result['diagnostics'][str(point)]
    loss_entry = next(e for e in result['training_log'] if e['step'] == point)
    print('SUMMARY', json.dumps({'step': point,
        'validation_loss_16': loss_entry['validation_loss_16'],
        'previous_512_validation_loss_16_at_2000': previous_512_run['validation_loss_16'],
        'training_seconds_excluding_evaluation_logging_and_checkpoints':
            loss_entry['training_seconds_excluding_evaluation_logging_and_checkpoints'],
        'mean_training_pair_presentations': point * BATCH_SIZE / len(train),
        'train_64': checked['train_64']['summary'],
        'validation_64': checked['validation_64']['summary']}), flush=True)
print('RESULT', json.dumps({'actual_steps': result['actual_steps'], 'run_finished': result['run_finished'],
    'training_examples': len(train), 'unique_train_questions': data_info['unique_train_questions'],
    'training_seconds_excluding_evaluation_logging_and_checkpoints': training_seconds,
    'active_wall_seconds_including_diagnostics_and_checkpointing':
        result['active_wall_seconds_including_diagnostics_and_checkpointing'],
    'checkpoint_files': result['checkpoint_files'], 'results_file': str(RESULTS_FILE)}), flush=True)
print('Save notebook with outputs and download results JSON/checkpoints.')
print('AST/exact reproduction do not establish functional correctness; test 1919 timing is not measured here.')
